# Part A3 — Wide-and-Deep Architecture

This notebook evaluates the required wide-and-deep model against the A1 baseline.

The final standardised prediction is

$$
y_{out} = y_{deep} + \lambda y_{wide}.
$$

The deep branch reuses the architecture selected in A1. The wide branch is one additional linear layer. Both branches receive the same concatenated embedding and continuous-feature representation.

### Connection to A1 and A2

- Hidden width = **256**
- Shared embedding dimension = **2**
- Final training duration = **10 epochs**
- Five evaluation seeds = **42, 43, 44, 45, 46**
- Combined-data preprocessing is imported from `src/parta_common.py`
- A1 baseline five-seed results are loaded from the output files produced by Part A2 rather than retrained

### Important note about lambda = 0

`lambda = 0` is a valid candidate. It mathematically removes the contribution of the wide branch from the prediction. However, it is **not required to reproduce the numerical A1 training trajectory exactly**, because the wide-and-deep model contains an additional layer and therefore consumes random numbers differently during model construction before mini-batch shuffling begins.

For the lambda search itself this is not a fairness problem: every lambda candidate uses the **same wide-and-deep architecture**, the same seed, the same initialisation sequence, and the same training procedure. The search must therefore be allowed to select whichever lambda has the lowest validation RMSE, including zero if that is genuinely best.

## 1. Environment and project setup

In [ ]:
from pathlib import Path
import sys
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader

ROOT = Path.cwd()
SRC = ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from config import Config
from data import prepare_data
from parta_common import prepare_combined_data, calculate_rmse, evaluate_regression

DATA_PATH = ROOT / "hdb_price_prediction.csv"
A2_OUTPUT_DIR = ROOT / "outputs" / "A2"
OUTPUT_DIR = ROOT / "outputs" / "A3"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Data exists:", DATA_PATH.exists())
print("Shared helper exists:", (SRC / "parta_common.py").exists())

### Environment verification

Part A3 should use the same project `.venv` and VS Code kernel as A1 and A2.

In [ ]:
from importlib.metadata import version as package_version

python_executable = Path(sys.executable).resolve()
project_venv = (ROOT / ".venv").resolve()

print("Python executable:", python_executable)
print("Python version:", sys.version.split()[0])
print("Project .venv:", project_venv)
print("Using project .venv:", str(project_venv).lower() in str(python_executable).lower())

for package in ["torch", "ray", "pandas", "numpy", "matplotlib"]:
    try:
        print(f"{package}: {package_version(package)}")
    except Exception:
        print(f"{package}: not installed")

## 2. Fixed A1 settings and A2 baseline results

Only lambda is tuned in A3. The A1-selected architecture and final training budget remain fixed.

The five-seed A1 baseline results are loaded from Part A2 so that the comparison uses exactly the same baseline runs.

In [ ]:
HIDDEN_WIDTH = 256
EMBEDDING_DIM = 2
A1_NUM_EPOCHS = 10

SELECTION_SEED = 42
SEEDS = [42, 43, 44, 45, 46]
LAMBDA_VALUES = [0, 0.25, 0.5, 1, 2]

A2_RESULTS_PATH = A2_OUTPUT_DIR / "A2_seed_level_results.csv"
A2_HISTORY_PATH = A2_OUTPUT_DIR / "A2_training_rmse_history.csv"

if not A2_RESULTS_PATH.exists() or not A2_HISTORY_PATH.exists():
    raise FileNotFoundError(
        "Run PartA2_final.ipynb first. The files "
        "outputs/A2/A2_seed_level_results.csv and "
        "outputs/A2/A2_training_rmse_history.csv are required."
    )

a2_results = pd.read_csv(A2_RESULTS_PATH)
a2_history = pd.read_csv(A2_HISTORY_PATH)

baseline_results = (
    a2_results[a2_results["variant"] == "baseline"]
    .sort_values("seed")
    .reset_index(drop=True)
)

baseline_history = (
    a2_history[a2_history["variant"] == "baseline"]
    .copy()
)

print("Hidden width:", HIDDEN_WIDTH)
print("Embedding dimension:", EMBEDDING_DIM)
print("Final training epochs:", A1_NUM_EPOCHS)
print("Lambda candidates:", LAMBDA_VALUES)
print("Seeds:", SEEDS)

display(baseline_results)

## 3. Wide-and-deep model

The concatenated input dimension is

$$
4 \times 2 + 6 = 14.
$$

The two branches are:

- Deep branch: `14 → 256 → 1`, with LayerNorm and ReLU
- Wide branch: `14 → 1`

No special RNG manipulation is used. Each lambda trial starts from the same seed and constructs the same wide-and-deep architecture, so all lambda candidates receive the same initialisation sequence and stochastic training protocol.

In [ ]:
class WideDeepModel(nn.Module):
    def __init__(
        self,
        cardinalities,
        target_mean,
        target_std,
        config,
        lambda_value
    ):
        super().__init__()

        self.embeddings = nn.ModuleList(
            nn.Embedding(cardinality, width)
            for cardinality, width in zip(
                cardinalities,
                config.embedding_dims
            )
        )

        input_width = (
            sum(config.embedding_dims)
            + len(config.continuous_features)
        )

        self.deep = nn.Sequential(
            nn.Linear(input_width, config.hidden_width),
            nn.LayerNorm(config.hidden_width),
            nn.ReLU(),
            nn.Linear(config.hidden_width, 1)
        )

        self.wide = nn.Linear(input_width, 1)

        self.lambda_value = float(lambda_value)
        self.target_mean = target_mean
        self.target_std = target_std

    def encode_features(self, categorical, continuous):
        embedded = [
            embedding(categorical[:, index])
            for index, embedding in enumerate(self.embeddings)
        ]

        return torch.cat(
            embedded + [continuous],
            dim=1
        )

    def forward(self, categorical, continuous):
        features = self.encode_features(
            categorical,
            continuous
        )

        deep_output = self.deep(features).squeeze(1)
        wide_output = self.wide(features).squeeze(1)

        return (
            deep_output
            + self.lambda_value * wide_output
        )

    def predict_price(self, categorical, continuous):
        return (
            self(categorical, continuous)
            * self.target_std
            + self.target_mean
        )

### Architecture checks

The additional wide branch should add only 15 parameters: 14 weights and 1 bias.

A second check verifies the mathematical meaning of `lambda = 0`: for a fixed untrained model and batch, the full model output must equal the deep-branch output exactly.

In [ ]:
check_config = Config(
    seed=SELECTION_SEED,
    hidden_width=HIDDEN_WIDTH,
    embedding_dims=(EMBEDDING_DIM,) * 4
)

check_data = prepare_data(
    DATA_PATH,
    check_config
)

torch.manual_seed(SELECTION_SEED)

check_model = WideDeepModel(
    check_data["cardinalities"],
    check_data["target_mean"],
    check_data["target_std"],
    check_config,
    lambda_value=0
)

wide_deep_params = sum(
    p.numel()
    for p in check_model.parameters()
    if p.requires_grad
)

wide_branch_params = sum(
    p.numel()
    for p in check_model.wide.parameters()
    if p.requires_grad
)

categorical, continuous, _ = next(
    iter(
        DataLoader(
            check_data["train"],
            batch_size=32,
            shuffle=False
        )
    )
)

with torch.no_grad():
    features = check_model.encode_features(
        categorical,
        continuous
    )

    full_output = check_model(
        categorical,
        continuous
    )

    deep_only_output = (
        check_model.deep(features)
        .squeeze(1)
    )

    max_difference = (
        full_output - deep_only_output
    ).abs().max().item()

print("Wide-and-deep trainable parameters:", wide_deep_params)
print("Parameters added by wide branch:", wide_branch_params)
print("Max |lambda=0 output - deep output|:", max_difference)

assert max_difference < 1e-7

## 4. Lambda search on the A1 train/validation split

The search follows the A1 model-selection protocol:

- 2017–2020 for training
- 2021 for validation
- Adam optimiser
- maximum 20 epochs
- patience 4
- minimum improvement `1e-4`
- selection seed 42

Every lambda candidate uses the same wide-and-deep architecture. Only the scalar lambda changes.

In [ ]:
def train_wide_deep_selection(
    lambda_value,
    seed=SELECTION_SEED,
    verbose=False
):
    config = Config(
        seed=seed,
        hidden_width=HIDDEN_WIDTH,
        embedding_dims=(EMBEDDING_DIM,) * 4
    )

    torch.manual_seed(seed)
    torch.use_deterministic_algorithms(
        True,
        warn_only=True
    )

    data = prepare_data(
        DATA_PATH,
        config
    )

    model = WideDeepModel(
        data["cardinalities"],
        data["target_mean"],
        data["target_std"],
        config,
        lambda_value
    )

    train_loader = DataLoader(
        data["train"],
        batch_size=config.batch_size,
        shuffle=True
    )

    loss_function = nn.MSELoss()

    optimiser = torch.optim.Adam(
        model.parameters(),
        lr=config.learning_rate
    )

    best_validation_rmse = float("inf")
    best_weights = None
    best_epoch = 0
    epochs_without_improvement = 0

    train_rmse_history = []
    validation_rmse_history = []

    for epoch in range(
        1,
        config.max_epochs + 1
    ):
        model.train()

        squared_error = 0.0
        row_count = 0

        for categorical, continuous, target in train_loader:
            optimiser.zero_grad()

            standard_prediction = model(
                categorical,
                continuous
            )

            standard_target = (
                target - model.target_mean
            ) / model.target_std

            loss = loss_function(
                standard_prediction,
                standard_target
            )

            loss.backward()
            optimiser.step()

            prediction = (
                standard_prediction.detach()
                * model.target_std
                + model.target_mean
            )

            squared_error += (
                (prediction - target)
                .square()
                .sum()
                .item()
            )

            row_count += len(target)

        train_rmse = (
            squared_error / row_count
        ) ** 0.5

        validation_rmse = calculate_rmse(
            model,
            data["validation"],
            config.batch_size
        )

        train_rmse_history.append(
            train_rmse
        )

        validation_rmse_history.append(
            validation_rmse
        )

        if verbose:
            print(
                f"lambda={lambda_value:<4} | "
                f"epoch {epoch:02d} | "
                f"train RMSE {train_rmse:,.2f} | "
                f"validation RMSE {validation_rmse:,.2f}"
            )

        if (
            best_weights is None
            or validation_rmse
            < best_validation_rmse
            - config.early_stopping_threshold
        ):
            best_validation_rmse = (
                validation_rmse
            )

            best_weights = copy.deepcopy(
                model.state_dict()
            )

            best_epoch = epoch
            epochs_without_improvement = 0

        else:
            epochs_without_improvement += 1

            if (
                epochs_without_improvement
                == config.early_stopping_patience
            ):
                if verbose:
                    print(
                        f"Early stopping after epoch {epoch}"
                    )
                break

    model.load_state_dict(
        best_weights
    )

    return {
        "lambda": float(lambda_value),
        "best_epoch": best_epoch,
        "best_validation_rmse": best_validation_rmse,
        "train_rmse_history": train_rmse_history,
        "validation_rmse_history": validation_rmse_history
    }

In [ ]:
lambda_search_results = []
lambda_search_histories = {}

for lambda_value in LAMBDA_VALUES:
    print(f"Running lambda = {lambda_value}")

    result = train_wide_deep_selection(
        lambda_value=lambda_value,
        seed=SELECTION_SEED,
        verbose=False
    )

    lambda_search_results.append({
        "lambda": result["lambda"],
        "best_epoch": result["best_epoch"],
        "best_validation_rmse": result["best_validation_rmse"]
    })

    lambda_search_histories[lambda_value] = {
        "train": result["train_rmse_history"],
        "validation": result["validation_rmse_history"]
    }

    print(
        f"best epoch = {result['best_epoch']} | "
        f"best validation RMSE = "
        f"{result['best_validation_rmse']:,.2f}"
    )

lambda_search_df = (
    pd.DataFrame(lambda_search_results)
    .sort_values("best_validation_rmse")
    .reset_index(drop=True)
)

lambda_search_df

### Select the optimal lambda

The candidate with the lowest **best validation RMSE** is selected automatically. The code does not force a non-zero value.

In [ ]:
best_lambda_row = lambda_search_df.iloc[0]

BEST_LAMBDA = float(
    best_lambda_row["lambda"]
)

BEST_LAMBDA_VALIDATION_RMSE = float(
    best_lambda_row["best_validation_rmse"]
)

BEST_LAMBDA_SEARCH_EPOCH = int(
    best_lambda_row["best_epoch"]
)

FINAL_NUM_EPOCHS = A1_NUM_EPOCHS

print("Selected lambda:", BEST_LAMBDA)
print(
    "Best validation RMSE:",
    f"{BEST_LAMBDA_VALIDATION_RMSE:,.2f}"
)
print(
    "Best epoch during lambda search:",
    BEST_LAMBDA_SEARCH_EPOCH
)
print(
    "Epochs used for final five-seed comparison:",
    FINAL_NUM_EPOCHS
)

if BEST_LAMBDA == 0:
    print(
        "\nNote: lambda = 0 is a valid experimental result. "
        "It means the wide branch did not improve validation RMSE "
        "under this search protocol."
    )

### Best validation RMSE across lambda values

In [ ]:
plot_search_df = (
    lambda_search_df
    .sort_values("lambda")
)

plt.figure(figsize=(7, 5))

plt.plot(
    plot_search_df["lambda"],
    plot_search_df["best_validation_rmse"],
    marker="o"
)

for _, row in plot_search_df.iterrows():
    plt.annotate(
        f'{row["best_validation_rmse"] / 1000:.1f}k',
        (
            row["lambda"],
            row["best_validation_rmse"]
        ),
        xytext=(0, 8),
        textcoords="offset points",
        ha="center"
    )

plt.xlabel("Lambda")
plt.ylabel("Best validation RMSE (SGD)")
plt.title("A3 Lambda Search")
plt.xticks(LAMBDA_VALUES)
plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "A3_lambda_search.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Validation learning curves for all lambda candidates

This additional plot makes the lambda selection easier to diagnose than looking only at the single best RMSE from each run.

In [ ]:
plt.figure(figsize=(8, 5))

for lambda_value in LAMBDA_VALUES:
    history = (
        lambda_search_histories[
            lambda_value
        ]["validation"]
    )

    epochs = range(
        1,
        len(history) + 1
    )

    plt.plot(
        epochs,
        history,
        marker="o",
        label=f"lambda = {lambda_value}"
    )

plt.xlabel("Epoch")
plt.ylabel("Validation RMSE (SGD)")
plt.title("Validation RMSE during Lambda Search")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "A3_lambda_validation_curves.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 5. Five-seed evaluation of the selected wide-and-deep model

The selected lambda is evaluated with the same five seeds used in Part A2.

For direct comparison with the A1 baseline:

- preprocessing is fit on combined 2017–2021 data
- A1 hidden width, embedding dimension, optimiser, learning rate and batch size are unchanged
- every run is trained for 10 epochs
- the 2022 test set is evaluated once after training

In [ ]:
def train_wide_deep_final(
    seed,
    lambda_value=BEST_LAMBDA,
    verbose=False
):
    config = Config(
        seed=seed,
        hidden_width=HIDDEN_WIDTH,
        embedding_dims=(EMBEDDING_DIM,) * 4
    )

    torch.manual_seed(seed)
    torch.use_deterministic_algorithms(
        True,
        warn_only=True
    )

    data = prepare_combined_data(
        DATA_PATH,
        config
    )

    model = WideDeepModel(
        data["cardinalities"],
        data["target_mean"],
        data["target_std"],
        config,
        lambda_value
    )

    train_loader = DataLoader(
        data["combined"],
        batch_size=config.batch_size,
        shuffle=True
    )

    loss_function = nn.MSELoss()

    optimiser = torch.optim.Adam(
        model.parameters(),
        lr=config.learning_rate
    )

    train_rmse_history = []

    for epoch in range(
        1,
        FINAL_NUM_EPOCHS + 1
    ):
        model.train()

        squared_error = 0.0
        row_count = 0

        for categorical, continuous, target in train_loader:
            optimiser.zero_grad()

            standard_prediction = model(
                categorical,
                continuous
            )

            standard_target = (
                target - model.target_mean
            ) / model.target_std

            loss = loss_function(
                standard_prediction,
                standard_target
            )

            loss.backward()
            optimiser.step()

            prediction = (
                standard_prediction.detach()
                * model.target_std
                + model.target_mean
            )

            squared_error += (
                (prediction - target)
                .square()
                .sum()
                .item()
            )

            row_count += len(target)

        train_rmse = (
            squared_error / row_count
        ) ** 0.5

        train_rmse_history.append(
            train_rmse
        )

        if verbose:
            print(
                f"seed {seed} | "
                f"epoch {epoch:02d} | "
                f"train RMSE {train_rmse:,.2f}"
            )

    test_rmse, test_r2 = evaluate_regression(
        model,
        data["test"],
        config.batch_size
    )

    return {
        "seed": seed,
        "lambda": float(lambda_value),
        "model": model,
        "test_rmse": test_rmse,
        "test_r2": test_r2,
        "train_rmse_history": train_rmse_history
    }

In [ ]:
wide_deep_results = []
wide_deep_training_histories = {}

for seed in SEEDS:
    result = train_wide_deep_final(
        seed=seed,
        lambda_value=BEST_LAMBDA,
        verbose=False
    )

    wide_deep_results.append({
        "seed": seed,
        "lambda": BEST_LAMBDA,
        "test_rmse": result["test_rmse"],
        "test_r2": result["test_r2"]
    })

    wide_deep_training_histories[
        seed
    ] = result[
        "train_rmse_history"
    ]

    if seed == 42:
        torch.save(
            result["model"].state_dict(),
            OUTPUT_DIR /
            "selected_wide_deep_seed42.pt"
        )

    print(
        f"seed {seed} | "
        f"test RMSE {result['test_rmse']:,.2f} | "
        f"test R² {result['test_r2']:.4f}"
    )

wide_deep_df = pd.DataFrame(
    wide_deep_results
)

wide_deep_df

## 6. Mean and sample SD compared with the A1 baseline

In [ ]:
def summarize_results(df, label):
    return {
        "model": label,
        "test_rmse_mean": df["test_rmse"].mean(),
        "test_rmse_sd": df["test_rmse"].std(ddof=1),
        "test_r2_mean": df["test_r2"].mean(),
        "test_r2_sd": df["test_r2"].std(ddof=1)
    }


comparison_summary_df = pd.DataFrame([
    summarize_results(
        baseline_results,
        "A1 baseline"
    ),
    summarize_results(
        wide_deep_df,
        f"Wide-and-deep (lambda={BEST_LAMBDA:g})"
    )
])

baseline_rmse_mean = (
    comparison_summary_df
    .loc[
        comparison_summary_df["model"]
        == "A1 baseline",
        "test_rmse_mean"
    ]
    .iloc[0]
)

baseline_r2_mean = (
    comparison_summary_df
    .loc[
        comparison_summary_df["model"]
        == "A1 baseline",
        "test_r2_mean"
    ]
    .iloc[0]
)

comparison_summary_df[
    "delta_rmse_vs_baseline"
] = (
    comparison_summary_df[
        "test_rmse_mean"
    ]
    - baseline_rmse_mean
)

comparison_summary_df[
    "delta_r2_vs_baseline"
] = (
    comparison_summary_df[
        "test_r2_mean"
    ]
    - baseline_r2_mean
)

comparison_summary_df

In [ ]:
report_table = (
    comparison_summary_df[
        ["model"]
    ]
    .copy()
)

report_table[
    "Test RMSE (mean ± SD)"
] = comparison_summary_df.apply(
    lambda row:
    f'{row["test_rmse_mean"]:,.0f} ± '
    f'{row["test_rmse_sd"]:,.0f}',
    axis=1
)

report_table[
    "Test R² (mean ± SD)"
] = comparison_summary_df.apply(
    lambda row:
    f'{row["test_r2_mean"]:.4f} ± '
    f'{row["test_r2_sd"]:.4f}',
    axis=1
)

report_table[
    "ΔRMSE vs baseline"
] = comparison_summary_df[
    "delta_rmse_vs_baseline"
].map(
    lambda value: f"{value:+,.0f}"
)

report_table[
    "ΔR² vs baseline"
] = comparison_summary_df[
    "delta_r2_vs_baseline"
].map(
    lambda value: f"{value:+.4f}"
)

report_table

## 7. Paired seed-wise comparison

Because both models use the same five seed values, paired differences show whether the direction of the effect is consistent across runs.

In [ ]:
paired_df = (
    baseline_results[
        ["seed", "test_rmse", "test_r2"]
    ]
    .rename(columns={
        "test_rmse": "baseline_rmse",
        "test_r2": "baseline_r2"
    })
    .merge(
        wide_deep_df[
            ["seed", "test_rmse", "test_r2"]
        ]
        .rename(columns={
            "test_rmse": "wide_deep_rmse",
            "test_r2": "wide_deep_r2"
        }),
        on="seed"
    )
)

paired_df["delta_rmse"] = (
    paired_df["wide_deep_rmse"]
    - paired_df["baseline_rmse"]
)

paired_df["delta_r2"] = (
    paired_df["wide_deep_r2"]
    - paired_df["baseline_r2"]
)

paired_df

In [ ]:
paired_summary_df = pd.DataFrame([{
    "mean_delta_rmse":
        paired_df["delta_rmse"].mean(),

    "sd_delta_rmse":
        paired_df["delta_rmse"].std(ddof=1),

    "mean_delta_r2":
        paired_df["delta_r2"].mean(),

    "sd_delta_r2":
        paired_df["delta_r2"].std(ddof=1),

    "wide_deep_better_rmse_seeds": int(
        (paired_df["delta_rmse"] < 0).sum()
    ),

    "wide_deep_better_r2_seeds": int(
        (paired_df["delta_r2"] > 0).sum()
    )
}])

paired_summary_df

## 8. Test RMSE across five seeds

In [ ]:
plot_models = [
    ("A1 baseline", baseline_results),
    (
        f"Wide-and-deep\nlambda={BEST_LAMBDA:g}",
        wide_deep_df
    )
]

x_positions = np.arange(
    len(plot_models)
)

jitter = np.linspace(
    -0.08,
    0.08,
    len(SEEDS)
)

plt.figure(figsize=(7, 5))

for x, (label, df) in zip(
    x_positions,
    plot_models
):
    values = (
        df.sort_values("seed")[
            "test_rmse"
        ]
        .to_numpy()
    )

    scatter = plt.scatter(
        x + jitter,
        values,
        s=45,
        alpha=0.85
    )

    plt.errorbar(
        x,
        values.mean(),
        yerr=values.std(ddof=1),
        fmt="D",
        markersize=8,
        capsize=5,
        linewidth=1.5,
        color=scatter.get_facecolor()[0]
    )

plt.xticks(
    x_positions,
    [
        label
        for label, _ in plot_models
    ]
)

plt.ylabel("Test RMSE (SGD)")
plt.title(
    "A3 Test RMSE across Five Seeds"
)
plt.grid(
    axis="y",
    alpha=0.25
)
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR /
    "A3_test_rmse_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 9. Test R² across five seeds

In [ ]:
plt.figure(figsize=(7, 5))

for x, (label, df) in zip(
    x_positions,
    plot_models
):
    values = (
        df.sort_values("seed")[
            "test_r2"
        ]
        .to_numpy()
    )

    scatter = plt.scatter(
        x + jitter,
        values,
        s=45,
        alpha=0.85
    )

    plt.errorbar(
        x,
        values.mean(),
        yerr=values.std(ddof=1),
        fmt="D",
        markersize=8,
        capsize=5,
        linewidth=1.5,
        color=scatter.get_facecolor()[0]
    )

plt.xticks(
    x_positions,
    [
        label
        for label, _ in plot_models
    ]
)

plt.ylabel("Test R²")
plt.title(
    "A3 Test R² across Five Seeds"
)
plt.grid(
    axis="y",
    alpha=0.25
)
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR /
    "A3_test_r2_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 10. Training RMSE: A1 baseline vs wide-and-deep

The baseline histories are loaded from A2. The wide-and-deep histories come from the five A3 runs. Lines show mean training RMSE and shaded bands show ±1 sample SD.

In [ ]:
epochs = np.arange(
    1,
    FINAL_NUM_EPOCHS + 1
)

baseline_history_matrix = np.array([
    (
        baseline_history[
            baseline_history["seed"] == seed
        ]
        .sort_values("epoch")[
            "train_rmse"
        ]
        .to_numpy()
    )
    for seed in SEEDS
])

wide_deep_history_matrix = np.array([
    wide_deep_training_histories[seed]
    for seed in SEEDS
])

plt.figure(figsize=(8, 5))

for label, history_matrix in [
    (
        "A1 baseline",
        baseline_history_matrix
    ),
    (
        f"Wide-and-deep "
        f"(lambda={BEST_LAMBDA:g})",
        wide_deep_history_matrix
    )
]:
    mean_history = (
        history_matrix.mean(axis=0)
    )

    sd_history = (
        history_matrix.std(
            axis=0,
            ddof=1
        )
    )

    line, = plt.plot(
        epochs,
        mean_history,
        marker="o",
        label=label
    )

    plt.fill_between(
        epochs,
        mean_history - sd_history,
        mean_history + sd_history,
        alpha=0.15,
        color=line.get_color()
    )

plt.xlabel("Epoch")
plt.ylabel("Training RMSE (SGD)")
plt.title(
    "Training RMSE on 2017–2021 "
    "across Five Seeds"
)
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR /
    "A3_training_rmse_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 11. Save A3 outputs

In [ ]:
history_rows = []

for seed, history in (
    wide_deep_training_histories.items()
):
    for epoch, train_rmse in enumerate(
        history,
        start=1
    ):
        history_rows.append({
            "seed": seed,
            "lambda": BEST_LAMBDA,
            "epoch": epoch,
            "train_rmse": train_rmse
        })

wide_deep_history_df = pd.DataFrame(
    history_rows
)

lambda_search_df.to_csv(
    OUTPUT_DIR /
    "A3_lambda_search.csv",
    index=False
)

wide_deep_df.to_csv(
    OUTPUT_DIR /
    "A3_seed_level_results.csv",
    index=False
)

comparison_summary_df.to_csv(
    OUTPUT_DIR /
    "A3_summary_mean_sd.csv",
    index=False
)

paired_df.to_csv(
    OUTPUT_DIR /
    "A3_paired_vs_baseline.csv",
    index=False
)

paired_summary_df.to_csv(
    OUTPUT_DIR /
    "A3_paired_summary.csv",
    index=False
)

wide_deep_history_df.to_csv(
    OUTPUT_DIR /
    "A3_training_rmse_history.csv",
    index=False
)

pd.DataFrame([{
    "selected_lambda":
        BEST_LAMBDA,
    "best_validation_rmse":
        BEST_LAMBDA_VALIDATION_RMSE,
    "lambda_search_best_epoch":
        BEST_LAMBDA_SEARCH_EPOCH,
    "final_training_epochs":
        FINAL_NUM_EPOCHS
}]).to_csv(
    OUTPUT_DIR /
    "A3_selected_configuration.csv",
    index=False
)

print(
    "Saved A3 outputs to:",
    OUTPUT_DIR
)

print(
    "Seed-42 model saved as:",
    OUTPUT_DIR /
    "selected_wide_deep_seed42.pt"
)

## 12. Interpretation guide

Use the actual values produced above.

### Mechanism

The deep branch models nonlinear relationships and feature interactions. The wide branch gives the model a direct linear path from the same 14-dimensional input representation to the prediction. If useful relationships are approximately additive, this branch can represent them directly while the deep branch focuses on nonlinear structure.

### Evidence to examine

- Compare the validation RMSE for all five lambda values.
- Compare the selected wide-and-deep model with the A1 baseline using mean ± sample SD of test RMSE and R².
- Check the paired seed-wise differences to see whether any improvement is consistent.
- Compare the training-RMSE curves to determine whether the added branch changes optimisation or mainly affects generalisation.

### Interpretation if lambda = 0 is selected

This is a valid outcome. It means the validation evidence does not support using the added linear contribution under the tested settings. Do not replace zero with a non-zero value simply because the architecture is called wide-and-deep.

### Limitations and confounds

1. Lambda is selected using one model-selection seed, so the chosen value may be seed-sensitive.
2. The wide-and-deep model has additional trainable parameters, so the comparison changes both architecture and model capacity.
3. The split is temporal: training uses earlier years and testing uses 2022, so temporal distribution shift may affect both architectures.